# 01. Search Papers

OpenAlex API로 키워드 검색 → `data/papers_raw.csv`로 저장.

**OpenAlex**는 무인증·무료·일일 100k 호출이고 모든 분야 + 인용 데이터를 줍니다. 가장 부담 없이 시작할 수 있습니다.

추후 arXiv, Semantic Scholar로 확장 가능 — 같은 노트북 안에 별도 셀로 추가하세요.

In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 본인 이메일을 넣으면 OpenAlex가 'polite pool'에 넣어 더 안정적인 응답을 줍니다.
MAILTO = os.environ.get('OPENALEX_MAILTO', 'student@example.com')

## 검색 파라미터

`topic_scoping(rr)`에서 받은 키워드와 연도 범위를 여기에 넣습니다.

In [ ]:
QUERY = 'agent architectures large language models'   # ← 본인 키워드로 교체
FROM_YEAR = 2020
TO_YEAR = 2026
MAX_RESULTS = 50

In [ ]:
def search_openalex(query, from_year, to_year, max_results=50, mailto=MAILTO):
    url = 'https://api.openalex.org/works'
    params = {
        'search': query,
        'filter': f'from_publication_date:{from_year}-01-01,to_publication_date:{to_year}-12-31',
        'per-page': min(max_results, 200),
        'mailto': mailto,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    results = r.json().get('results', [])

    rows = []
    for w in results[:max_results]:
        authors = ', '.join(a['author']['display_name'] for a in w.get('authorships', [])[:5])
        rows.append({
            'id': w.get('id'),
            'title': w.get('title'),
            'authors': authors,
            'year': w.get('publication_year'),
            'venue': (w.get('primary_location') or {}).get('source', {}).get('display_name') if w.get('primary_location') else None,
            'cited_by_count': w.get('cited_by_count', 0),
            'doi': w.get('doi'),
            'oa_url': (w.get('open_access') or {}).get('oa_url'),
            'abstract': _reconstruct_abstract(w.get('abstract_inverted_index')),
        })
    return pd.DataFrame(rows)


def _reconstruct_abstract(inv_index):
    if not inv_index:
        return None
    positions = []
    for word, idxs in inv_index.items():
        for i in idxs:
            positions.append((i, word))
    positions.sort()
    return ' '.join(w for _, w in positions)

df = search_openalex(QUERY, FROM_YEAR, TO_YEAR, MAX_RESULTS)
print(f'{len(df)} papers fetched')
df.head()

In [ ]:
out = DATA_DIR / 'papers_raw.csv'
df.to_csv(out, index=False)
print(f'Saved → {out.resolve()}')

## 다음 단계

`02_score_quality.ipynb`을 열어 인용수·연도 기반 정량 점수를 계산하세요.